In [ ]:
!pip uninstall -y tensorflow tensorflow-cpu keras
!pip install -q "unbabel-comet==2.2.7"

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
Found existing installation: keras 3.13.2
Uninstalling keras-3.13.2:
  Successfully uninstalled keras-3.13.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.18 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.


In [ ]:
import torch
import comet
import numpy as np
import google.protobuf

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("NumPy:", np.__version__)
print("Protobuf:", google.protobuf.__version__)
print("COMET imported successfully")

/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


PyTorch: 2.11.0+cu128
CUDA available: True
NumPy: 1.26.4
Protobuf: 5.29.6
COMET imported successfully


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Ours_data")

SRC_FILE = DATA_DIR / "test.en"
REF_FILE = DATA_DIR / "test.vi"
SAMPLE_PRED_FILE = DATA_DIR / "hymt.vi"


def read_lines(path):
    with open(path, "r", encoding="utf-8") as file:
        return [line.rstrip("\r\n") for line in file]


src_sentences = read_lines(SRC_FILE)
ref_sentences = read_lines(REF_FILE)
sample_predictions = read_lines(SAMPLE_PRED_FILE)

sample_size = len(sample_predictions)

sample_sources = src_sentences[:sample_size]
sample_references = ref_sentences[:sample_size]

print("Nguồn:", len(sample_sources))
print("Dự đoán:", len(sample_predictions))
print("Tham chiếu:", len(sample_references))

assert sample_size > 0, "prediction_sample.vi đang rỗng"
assert len(sample_sources) == sample_size
assert len(sample_references) == sample_size
assert not any(not item.strip() for item in sample_predictions), (
    "Có bản dịch rỗng trong prediction_sample.vi"
)

Nguồn: 889
Dự đoán: 889
Tham chiếu: 889


In [ ]:
comet_data = [
    {
        "src": source,
        "mt": prediction,
        "ref": reference,
    }
    for source, prediction, reference in zip(
        sample_sources,
        sample_predictions,
        sample_references,
    )
]

print("Số mẫu COMET:", len(comet_data))
print(comet_data[0])

Số mẫu COMET: 889
{'src': '"Generative Pre-trained Transformer models, known as GPT or OPT, set themselves apart through breakthrough performance across complex language modelling tasks, but also by their extremely high computational and storage costs. Specifically, due to their massive size, even inference for large, highly-accurate GPT models may require multiple performant GPUs, which limits the usability of such models. While there is emerging work on relieving this pressure via model compression, the applicability and performance of existing compression techniques is limited by the scale and complexity of GPT models. In this paper, we address this challenge, and propose GPTQ, a new one-shot weight quantization method based on approximate second-order information, that is both highly-accurate and highly-efficient. Specifically, GPTQ can quantize GPT models with 175 billion parameters in approximately four GPU hours, reducing the bitwidth down to 3 or 4 bits per weight, with negligi

In [ ]:
import torch
from comet import download_model, load_from_checkpoint

COMET_MODEL = "Unbabel/wmt22-comet-da"

model_path = download_model(COMET_MODEL)
comet_model = load_from_checkpoint(model_path)

print("Đã load:", COMET_MODEL)
print("CUDA:", torch.cuda.is_available())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

Đã load: Unbabel/wmt22-comet-da
CUDA: True


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [ ]:
gpus = 1 if torch.cuda.is_available() else 0

comet_output = comet_model.predict(
    comet_data,
    batch_size=8,
    gpus=gpus,
    num_workers=2,
    progress_bar=True,
)

system_score = float(comet_output.system_score)
sentence_scores = [
    float(score)
    for score in comet_output.scores
]

print("Số câu đã đánh giá:", len(sentence_scores))
print(f"COMET system score: {system_score:.6f}")
print("5 sentence scores đầu:", sentence_scores)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/

Số câu đã đánh giá: 889
COMET system score: 0.820714
5 sentence scores đầu: [0.9067816734313965, 0.8736305832862854, 0.8804700970649719, 0.865236759185791, 0.8764067888259888, 0.8665433526039124, 0.8732951283454895, 0.8591142296791077, 0.8784314393997192, 0.8689269423484802, 0.8660319447517395, 0.7900699973106384, 0.844245195388794, 0.8495989441871643, 0.8666355013847351, 0.8675746321678162, 0.8563821911811829, 0.8567004799842834, 0.8821106553077698, 0.8463173508644104, 0.8631872534751892, 0.8724221587181091, 0.8728986978530884, 0.8314502239227295, 0.8795877695083618, 0.8540745973587036, 0.8761625289916992, 0.8354588150978088, 0.8722805380821228, 0.8602960109710693, 0.8595018982887268, 0.8937791585922241, 0.8277943134307861, 0.8394576907157898, 0.8615379333496094, 0.8483934998512268, 0.8726812601089478, 0.8721169829368591, 0.885436475276947, 0.8556908965110779, 0.9074820876121521, 0.8511313796043396, 0.8268032670021057, 0.8335866928100586, 0.745542585849762, 0.8522419333457947, 0.88811

Ben bleu

In [ ]:
!pip install -q sacrebleu

In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Ours_data")

SRC_FILE = DATA_DIR / "test.en"
REF_FILE = DATA_DIR / "test.vi"

# Benchmark 100 câu:
PRED_FILE = DATA_DIR / "hymt.vi"

# Khi chạy toàn bộ tập test, thay dòng trên bằng:
# PRED_FILE = DATA_DIR / "gemma.vi"


def read_lines(path):
    with open(path, "r", encoding="utf-8") as file:
        return [line.rstrip("\r\n") for line in file]


sources_all = read_lines(SRC_FILE)
references_all = read_lines(REF_FILE)
predictions = read_lines(PRED_FILE)

sample_size = len(predictions)
sources = sources_all[:sample_size]
references = references_all[:sample_size]

print("File dự đoán:", PRED_FILE)
print("Số câu nguồn:", len(sources))
print("Số câu dự đoán:", len(predictions))
print("Số câu tham chiếu:", len(references))

assert sample_size > 0, "File dự đoán đang rỗng"
assert len(sources) == len(predictions) == len(references)
assert not any(
    source.strip() and not prediction.strip()
    for source, prediction in zip(sources, predictions)
), "Có bản dịch rỗng"

File dự đoán: /content/drive/MyDrive/Ours_data/hymt.vi
Số câu nguồn: 889
Số câu dự đoán: 889
Số câu tham chiếu: 889


In [ ]:
from sacrebleu.metrics import BLEU, CHRF

bleu_metric = BLEU(tokenize="13a")
chrf_metric = CHRF(
    char_order=6,
    word_order=2,
    beta=2,
)

bleu_result = bleu_metric.corpus_score(
    predictions,
    [references],
)

chrf_result = chrf_metric.corpus_score(
    predictions,
    [references],
)

bleu_score = float(bleu_result.score)
chrf_score = float(chrf_result.score)

print(f"BLEU:   {bleu_score:.4f}")
print(f"chrF++: {chrf_score:.4f}")

print("\nChi tiết BLEU:")
print(bleu_result)

print("\nChi tiết chrF++:")
print(chrf_result)

print("\nBLEU signature:")
print(bleu_metric.get_signature())

BLEU:   29.0702
chrF++: 55.3796

Chi tiết BLEU:
BLEU = 29.07 57.3/34.9/22.7/15.7 (BP = 1.000 ratio = 1.199 hyp_len = 168978 ref_len = 140948)

Chi tiết chrF++:
chrF2++ = 55.38

BLEU signature:
nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0


BEN bleurt


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content

!rm -rf bleurt
!git clone https://github.com/google-research/bleurt.git
%cd /content/bleurt
!pip install -q .

/content
Cloning into 'bleurt'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 134 (delta 0), reused 17 (delta 0), pack-reused 116 (from 1)
Receiving objects: 100% (134/134), 31.28 MiB | 35.78 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/bleurt
  Preparing metadata (setup.py) ... done


In [ ]:
%cd /content

!wget -q --show-progress \
    https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip

!unzip -q -o BLEURT-20.zip -d /content

/content
BLEURT-20.zip       100%[===================>]   1.99G  40.8MB/s    in 62s     


In [ ]:
from pathlib import Path

CHECKPOINT = Path("/content/BLEURT-20")

print("Checkpoint tồn tại:", CHECKPOINT.exists())
print("Các file:", [p.name for p in CHECKPOINT.iterdir()])

Checkpoint tồn tại: True
Các file: ['sent_piece.model', 'bleurt_config.json', 'bert_config.json', 'saved_model.pb', 'variables', 'sent_piece.vocab']


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Ours_data")

REFERENCE_FILE = DATA_DIR / "test.vi"

# Chạy thử:
PRED_FILE = DATA_DIR / "hymt.vi"

# Khi chấm toàn bộ, thay dòng trên bằng:
# PRED_FILE = DATA_DIR / "prediction.vi"

OUTPUT_DIR = DATA_DIR / "evaluation_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Reference:", REFERENCE_FILE)
print("Candidate:", PRED_FILE)
print("Output:", OUTPUT_DIR)

Reference: /content/drive/MyDrive/Ours_data/test.vi
Candidate: /content/drive/MyDrive/Ours_data/hymt.vi
Output: /content/drive/MyDrive/Ours_data/evaluation_results


In [ ]:
def read_lines(path: Path) -> list[str]:
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")

    return path.read_text(
        encoding="utf-8-sig"
    ).splitlines()


references = read_lines(REFERENCE_FILE)
predictions = read_lines(PRED_FILE)

print("Số câu reference:", len(references))
print("Số câu candidate:", len(predictions))

if len(references) != len(predictions):
    raise ValueError(
        "Reference và candidate không có cùng số dòng: "
        f"{len(references)} != {len(predictions)}"
    )

for line_id, (reference, prediction) in enumerate(
    zip(references, predictions),
    start=1
):
    if not reference.strip():
        raise ValueError(
            f"Reference rỗng tại dòng {line_id}"
        )

    if not prediction.strip():
        raise ValueError(
            f"Candidate rỗng tại dòng {line_id}"
        )

references = [line.strip() for line in references]
predictions = [line.strip() for line in predictions]

print("Dữ liệu hợp lệ.")

Số câu reference: 889
Số câu candidate: 889
Dữ liệu hợp lệ.


In [ ]:
!pip install -q --force-reinstall --no-cache-dir "numpy==1.26.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 391.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-h

In [ ]:
import numpy as np
import tensorflow as tf
from bleurt import score

CHECKPOINT = "/content/BLEURT-20"

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

scorer = score.LengthBatchingBleurtScorer(CHECKPOINT)

bleurt_scores = scorer.score(
    references=references,
    candidates=predictions,
    batch_size=16,
)

bleurt_scores = np.asarray(
    bleurt_scores,
    dtype=float
)

print("Đã chấm:", len(bleurt_scores), "câu")
print("BLEURT trung bình:", bleurt_scores.mean())

TensorFlow version: 2.20.0
GPU: []


Ben metricx

In [ ]:
%cd /content

!rm -rf metricx
!git clone https://github.com/google-research/metricx.git

!pip install -q --upgrade pip setuptools wheel
!pip install -q \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "sentencepiece>=0.2.0" \
    "accelerate>=0.34.0" \
    "safetensors>=0.4.5"

/content
Cloning into 'metricx'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 32 (delta 11), reused 3 (delta 3), pack-reused 12 (from 1)
Receiving objects: 100% (32/32), 22.87 KiB | 3.81 MiB/s, done.
Resolving deltas: 100% (14/14), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the package

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Ours_data")

SOURCE_FILE = DATA_DIR / "test.en"
HYPOTHESIS_FILE = DATA_DIR / "hymt.vi"
REFERENCE_FILE = DATA_DIR / "test.vi"

INPUT_JSONL = DATA_DIR / "hymt_metricx_input.jsonl"
OUTPUT_JSONL = DATA_DIR / "hymt_metricx_output.jsonl"
RESULT_CSV = DATA_DIR / "hymt_metricx_results.csv"

In [ ]:
for label, path in {
    "Source": SOURCE_FILE,
    "Prediction": HYPOTHESIS_FILE,
    "Reference": REFERENCE_FILE,
}.items():
    if not path.exists():
        raise FileNotFoundError(f"{label}: không tìm thấy {path}")

    print(f"{label}: {path.name} — {path.stat().st_size:,} bytes")

Source: test.en — 517,820 bytes
Prediction: hymt.vi — 948,031 bytes
Reference: test.vi — 769,753 bytes


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Ours_data")

SOURCE_FILE = DATA_DIR / "test.en"
PREDICTION_FILE = DATA_DIR / "hymt.vi"
REFERENCE_FILE = DATA_DIR / "test.vi"


def read_lines(path):
    with path.open("r", encoding="utf-8-sig") as file:
        return [line.strip() for line in file if line.strip()]


sources = read_lines(SOURCE_FILE)
predictions = read_lines(PREDICTION_FILE)
references = read_lines(REFERENCE_FILE)

print("Source:    ", len(sources))
print("Prediction:", len(predictions))
print("Reference: ", len(references))

if not (len(sources) == len(predictions) == len(references)):
    raise ValueError(
        "Số mẫu không bằng nhau. Không được chạy MetricX vì dữ liệu có thể bị lệch."
    )

print("\nBa file khớp số mẫu.")
print("\nMẫu đầu tiên:")
print("Source:    ", sources[0])
print("Prediction:", predictions[0])
print("Reference: ", references[0])

Source:     889
Prediction: 889
Reference:  889

Ba file khớp số mẫu.

Mẫu đầu tiên:
Source:     "Generative Pre-trained Transformer models, known as GPT or OPT, set themselves apart through breakthrough performance across complex language modelling tasks, but also by their extremely high computational and storage costs. Specifically, due to their massive size, even inference for large, highly-accurate GPT models may require multiple performant GPUs, which limits the usability of such models. While there is emerging work on relieving this pressure via model compression, the applicability and performance of existing compression techniques is limited by the scale and complexity of GPT models. In this paper, we address this challenge, and propose GPTQ, a new one-shot weight quantization method based on approximate second-order information, that is both highly-accurate and highly-efficient. Specifically, GPTQ can quantize GPT models with 175 billion parameters in approximately four GPU hou

In [ ]:
import json

INPUT_JSONL = DATA_DIR / "hymt_metricx_input.jsonl"

with INPUT_JSONL.open("w", encoding="utf-8") as file:
    for index, (source, prediction, reference) in enumerate(
        zip(sources, predictions, references)
    ):
        sample = {
            "id": index,
            "source": source,
            "hypothesis": prediction,
            "reference": reference,
        }
        file.write(json.dumps(sample, ensure_ascii=False) + "\n")

print("Đã tạo:", INPUT_JSONL)
print("Số mẫu:", len(sources))

Đã tạo: /content/drive/MyDrive/Ours_data/hymt_metricx_input.jsonl
Số mẫu: 889


In [ ]:
!pip install -q --upgrade "datasets==5.0.1"

In [ ]:
!pip uninstall -y wandb

Found existing installation: wandb 0.28.0
Uninstalling wandb-0.28.0:
  Successfully uninstalled wandb-0.28.0


In [ ]:
from pathlib import Path
import subprocess
import sys
import os

DATA_DIR = Path("/content/drive/MyDrive/Ours_data")
INPUT_JSONL = DATA_DIR / "hymt_metricx_input.jsonl"
OUTPUT_JSONL = DATA_DIR / "hymt_metricx_output.jsonl"

run_env = os.environ.copy()
run_env["WANDB_DISABLED"] = "true"
run_env["WANDB_MODE"] = "disabled"

command = [
    sys.executable,
    "-m", "metricx24.predict",
    "--tokenizer", "google/mt5-large",
    "--model_name_or_path",
    "google/metricx-24-hybrid-large-v2p6-bfloat16",
    "--max_input_length", "1536",
    "--batch_size", "1",
    "--input_file", str(INPUT_JSONL),
    "--output_file", str(OUTPUT_JSONL),
]

result = subprocess.run(
    command,
    cwd="/content/metricx",
    env=run_env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)

if result.returncode != 0:
    raise RuntimeError(f"MetricX thất bại, exit code: {result.returncode}")

print("Hoàn thành:", OUTPUT_JSONL)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(

Generating test split: 0 examples [00:0

In [ ]:
from pathlib import Path
import json

OUTPUT_JSONL = Path(
    "/content/drive/MyDrive/Ours_data/metricx_output.jsonl"
)

print("File tồn tại:", OUTPUT_JSONL.exists())

if not OUTPUT_JSONL.exists():
    raise FileNotFoundError(f"Không tìm thấy {OUTPUT_JSONL}")

print("Kích thước:", OUTPUT_JSONL.stat().st_size, "bytes")

with OUTPUT_JSONL.open("r", encoding="utf-8") as file:
    results = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

print("Số kết quả:", len(results))
print("\nKết quả đầu tiên:")
print(json.dumps(results[0], ensure_ascii=False, indent=2))

File tồn tại: True
Kích thước: 3228131 bytes
Số kết quả: 889

Kết quả đầu tiên:
{
  "id": 0,
  "source": "\"Generative Pre-trained Transformer models, known as GPT or OPT, set themselves apart through breakthrough performance across complex language modelling tasks, but also by their extremely high computational and storage costs. Specifically, due to their massive size, even inference for large, highly-accurate GPT models may require multiple performant GPUs, which limits the usability of such models. While there is emerging work on relieving this pressure via model compression, the applicability and performance of existing compression techniques is limited by the scale and complexity of GPT models. In this paper, we address this challenge, and propose GPTQ, a new one-shot weight quantization method based on approximate second-order information, that is both highly-accurate and highly-efficient. Specifically, GPTQ can quantize GPT models with 175 billion parameters in approximately fo

In [ ]:
import numpy as np

scores = np.array(
    [float(row["prediction"]) for row in results],
    dtype=float,
)

print(f"Số mẫu:             {len(scores):,}")
print(f"MetricX trung bình:  {scores.mean():.4f}")
print(f"Trung vị:            {np.median(scores):.4f}")
print(f"Độ lệch chuẩn:       {scores.std():.4f}")
print(f"Điểm tốt nhất:       {scores.min():.4f}")
print(f"Điểm tệ nhất:        {scores.max():.4f}")
print(f"Phân vị 25%:         {np.percentile(scores, 25):.4f}")
print(f"Phân vị 75%:         {np.percentile(scores, 75):.4f}")

Số mẫu:             889
MetricX trung bình:  5.4452
Trung vị:            5.0312
Độ lệch chuẩn:       2.6999
Điểm tốt nhất:       0.3750
Điểm tệ nhất:        16.1250
Phân vị 25%:         3.4844
Phân vị 75%:         7.0625
